# Machine Learning for Predicting Sediment Particle Size Distributions

## Mini Project — Reproduce and Extend

**Part 1:** Reproduce Random Forest and SVR from the reference paper.  
**Part 2:** Extend the work using XGBoost Regression.


### Reference settings
- Final features: `S`, `ub`, `np`, `T`
- Targets: `d50`, `sigma2` (σ²)
- Random Forest: 32 trees, 70/30 train-test split
- SVR: 70/15/15 train/CV/test, `C=2048`, `epsilon=4`
- Metric: R²
- Extension: XGBoost Regression


In [ ]:
!pip -q install xgboost

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from google.colab import files
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.metrics import r2_score
from xgboost import XGBRegressor

import warnings
warnings.filterwarnings('ignore')


## 1. Upload the dataset

In [ ]:
uploaded = files.upload()
filename = list(uploaded.keys())[0]
df = pd.read_csv(filename)

print('Dataset shape:', df.shape)
display(df.head())


## 2. Inspect the dataset

In [ ]:
print('Rows:', df.shape[0])
print('Columns:', df.shape[1])
print('\nColumns:')
print(df.columns.tolist())

print('\nMissing values:')
display(df.isnull().sum())

print('\nData types:')
df.info()


## 3. Define features and targets

The reference paper selects the four most important features: `S`, `ub`, `np`, and `T`.

In [ ]:
features = ['S', 'ub', 'np', 'T']
target_d50 = 'd50'
target_sigma2 = 'sigma2'

required_columns = features + [target_d50, target_sigma2]
missing_columns = [c for c in required_columns if c not in df.columns]

if missing_columns:
    raise ValueError(f'Missing required columns: {missing_columns}')

# Keep only complete rows for the variables used by the models.
model_df = df[required_columns].dropna().copy()

X = model_df[features]
y_d50 = model_df[target_d50]
y_sigma2 = model_df[target_sigma2]

print('Original rows:', len(df))
print('Rows used for modelling:', len(model_df))
print('Features:', features)
print('Targets:', target_d50, 'and', target_sigma2)


# PART 1 — Reproduce the Reference

## 4. Random Forest

Reference setting: 32 estimators and a 70/30 train-test split.

In [ ]:
X_train, X_test, y_d50_train, y_d50_test, y_sigma2_train, y_sigma2_test = train_test_split(
    X, y_d50, y_sigma2,
    test_size=0.30,
    random_state=42
)

print('Training samples:', len(X_train))
print('Testing samples:', len(X_test))


In [ ]:
rf_d50 = RandomForestRegressor(
    n_estimators=32,
    random_state=42
)
rf_d50.fit(X_train, y_d50_train)
rf_d50_pred = rf_d50.predict(X_test)
rf_d50_r2 = r2_score(y_d50_test, rf_d50_pred)

rf_sigma2 = RandomForestRegressor(
    n_estimators=32,
    random_state=42
)
rf_sigma2.fit(X_train, y_sigma2_train)
rf_sigma2_pred = rf_sigma2.predict(X_test)
rf_sigma2_r2 = r2_score(y_sigma2_test, rf_sigma2_pred)

print(f'Random Forest — d50 R²: {rf_d50_r2:.4f}')
print(f'Random Forest — sigma² R²: {rf_sigma2_r2:.4f}')


## 5. Random Forest feature importance

In [ ]:
importance = pd.DataFrame({
    'Feature': features,
    'Importance_d50': rf_d50.feature_importances_,
    'Importance_sigma2': rf_sigma2.feature_importances_
})

display(importance)

plt.figure(figsize=(8, 5))
plt.bar(importance['Feature'], importance['Importance_d50'])
plt.xlabel('Feature')
plt.ylabel('Importance')
plt.title('Random Forest Feature Importance for d50')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


## 6. Random Forest prediction plots

In [ ]:
plt.figure(figsize=(7, 6))
plt.scatter(y_d50_test, rf_d50_pred)
lims = [min(y_d50_test.min(), rf_d50_pred.min()), max(y_d50_test.max(), rf_d50_pred.max())]
plt.plot(lims, lims, linestyle='--')
plt.xlabel('True d50')
plt.ylabel('Predicted d50')
plt.title(f'Random Forest — d50 (R² = {rf_d50_r2:.3f})')
plt.tight_layout()
plt.show()

plt.figure(figsize=(7, 6))
plt.scatter(y_sigma2_test, rf_sigma2_pred)
lims = [min(y_sigma2_test.min(), rf_sigma2_pred.min()), max(y_sigma2_test.max(), rf_sigma2_pred.max())]
plt.plot(lims, lims, linestyle='--')
plt.xlabel('True σ²')
plt.ylabel('Predicted σ²')
plt.title(f'Random Forest — σ² (R² = {rf_sigma2_r2:.3f})')
plt.tight_layout()
plt.show()


## 7. SVR

Reference setting: 70% training, 15% cross-validation, 15% testing, `C=2048`, `epsilon=4`.

In [ ]:
# One common split is created and used for both SVR targets.
X_train_svr, X_temp_svr, y_d50_train_svr, y_d50_temp_svr, y_sigma2_train_svr, y_sigma2_temp_svr = train_test_split(
    X, y_d50, y_sigma2,
    test_size=0.30,
    random_state=42
)

X_cv_svr, X_test_svr, y_d50_cv, y_d50_test_svr, y_sigma2_cv, y_sigma2_test_svr = train_test_split(
    X_temp_svr, y_d50_temp_svr, y_sigma2_temp_svr,
    test_size=0.50,
    random_state=42
)

print('SVR training:', len(X_train_svr))
print('SVR CV:', len(X_cv_svr))
print('SVR test:', len(X_test_svr))


In [ ]:
svr_d50 = SVR(C=2048, epsilon=4)
svr_d50.fit(X_train_svr, y_d50_train_svr)
svr_d50_pred = svr_d50.predict(X_test_svr)
svr_d50_r2 = r2_score(y_d50_test_svr, svr_d50_pred)

svr_sigma2 = SVR(C=2048, epsilon=4)
svr_sigma2.fit(X_train_svr, y_sigma2_train_svr)
svr_sigma2_pred = svr_sigma2.predict(X_test_svr)
svr_sigma2_r2 = r2_score(y_sigma2_test_svr, svr_sigma2_pred)

print(f'SVR — d50 R²: {svr_d50_r2:.4f}')
print(f'SVR — sigma² R²: {svr_sigma2_r2:.4f}')


# PART 2 — Additional Model: XGBoost

XGBoost is not used in the reference paper. It is added as the extension model. The same RF 70/30 split is used for a fair comparison.

In [ ]:
xgb_params = dict(
    n_estimators=100,
    max_depth=3,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    objective='reg:squarederror'
)

xgb_d50 = XGBRegressor(**xgb_params)
xgb_d50.fit(X_train, y_d50_train)
xgb_d50_pred = xgb_d50.predict(X_test)
xgb_d50_r2 = r2_score(y_d50_test, xgb_d50_pred)

xgb_sigma2 = XGBRegressor(**xgb_params)
xgb_sigma2.fit(X_train, y_sigma2_train)
xgb_sigma2_pred = xgb_sigma2.predict(X_test)
xgb_sigma2_r2 = r2_score(y_sigma2_test, xgb_sigma2_pred)

print(f'XGBoost — d50 R²: {xgb_d50_r2:.4f}')
print(f'XGBoost — sigma² R²: {xgb_sigma2_r2:.4f}')


## 8. XGBoost prediction plots

In [ ]:
plt.figure(figsize=(7, 6))
plt.scatter(y_d50_test, xgb_d50_pred)
lims = [min(y_d50_test.min(), xgb_d50_pred.min()), max(y_d50_test.max(), xgb_d50_pred.max())]
plt.plot(lims, lims, linestyle='--')
plt.xlabel('True d50')
plt.ylabel('Predicted d50')
plt.title(f'XGBoost — d50 (R² = {xgb_d50_r2:.3f})')
plt.tight_layout()
plt.show()

plt.figure(figsize=(7, 6))
plt.scatter(y_sigma2_test, xgb_sigma2_pred)
lims = [min(y_sigma2_test.min(), xgb_sigma2_pred.min()), max(y_sigma2_test.max(), xgb_sigma2_pred.max())]
plt.plot(lims, lims, linestyle='--')
plt.xlabel('True σ²')
plt.ylabel('Predicted σ²')
plt.title(f'XGBoost — σ² (R² = {xgb_sigma2_r2:.3f})')
plt.tight_layout()
plt.show()


# 9. Final comparison — Part 1 + Part 2

In [ ]:
results = pd.DataFrame({
    'Model': [
        'Random Forest', 'Random Forest',
        'SVR', 'SVR',
        'XGBoost', 'XGBoost'
    ],
    'Target': ['d50', 'sigma²', 'd50', 'sigma²', 'd50', 'sigma²'],
    'Reported_R2': [0.81, 0.75, 0.76, 0.76, np.nan, np.nan],
    'Our_R2': [
        rf_d50_r2, rf_sigma2_r2,
        svr_d50_r2, svr_sigma2_r2,
        xgb_d50_r2, xgb_sigma2_r2
    ]
})

results['Difference'] = results['Our_R2'] - results['Reported_R2']

display(results.round(4))


## 10. Save results

The generated CSV files can be included with the final project submission.

In [ ]:
results.to_csv('final_model_results.csv', index=False)
importance.to_csv('feature_importance.csv', index=False)

print('Saved: final_model_results.csv')
print('Saved: feature_importance.csv')

files.download('final_model_results.csv')
files.download('feature_importance.csv')
